## Task 1: Data Exploration and Preprocessing

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# 1. Load Dataset
df = pd.read_csv('Alphabets_data.csv')

# 2. Data Exploration
print("Dataset Dimensions:", df.shape)
print("\nFirst 5 rows:")
print(df.head())

print("\nData Types & Non-Null Counts:")
print(df.info())

print("\nMissing Values Count:")
print(df.isnull().sum())

print("\nClass Distribution:")
print(df['letter'].value_counts())

# 3. Separate Features and Target
X = df.drop(columns=['letter'])
y = df['letter']

# 4. Encode Categorical Target Labels (A-Z -> 0-25)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# 5. Train-Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.20, random_state=42, stratify=y_encoded
)

# 6. Feature Normalization
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Task 2: Model Implementation (Baseline Model)

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# Set seeds for reproducibility
tf.random.set_seed(42)
np.random.seed(42)

# 1. Construct Baseline ANN Model
base_model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dense(26, activation='softmax')  # 26 classes for A-Z
])

# 2. Compile Model
base_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# 3. Train Model
history_base = base_model.fit(
    X_train_scaled, y_train,
    epochs=30,
    batch_size=32,
    validation_split=0.10,
    verbose=1
)

# 4. Baseline Model Evaluation
base_loss, base_acc = base_model.evaluate(X_test_scaled, y_test)
print(f"\nBaseline Test Accuracy: {base_acc:.4f}")

## Task 3: Hyperparameter Tuning

In [ ]:
!pip install keras-tuner



In [ ]:
import keras_tuner as kt

# 1. Define Model Builder Function for KerasTuner
def build_tunable_model(hp):
    model = Sequential()

    # Input Layer & First Hidden Layer
    hp_units = hp.Int('units_1', min_value=64, max_value=256, step=64)
    model.add(Dense(units=hp_units, activation=hp.Choice('act_1', values=['relu', 'elu']), input_shape=(X_train_scaled.shape[1],)))

    # Additional Hidden Layers
    for i in range(hp.Int('num_layers', 1, 3)):
        model.add(Dense(
            units=hp.Int(f'units_{i+2}', min_value=64, max_value=256, step=64),
            activation=hp.Choice(f'act_{i+2}', values=['relu', 'elu'])
        ))

    # Output Layer
    model.add(Dense(26, activation='softmax'))

    # Learning Rate Choice
    hp_lr = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=hp_lr),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# 2. Setup Random Search Tuner
tuner = kt.RandomSearch(
    build_tunable_model,
    objective='val_accuracy',
    max_trials=10,
    executions_per_trial=1,
    directory='tuning_dir',
    project_name='alphabet_ann'
)

# 3. Execute Hyperparameter Search
tuner.search(X_train_scaled, y_train, epochs=20, validation_split=0.1, verbose=1)

# Get Best Hyperparameters
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print("\nBest Hyperparameters Found:")
print(f"Layers: {best_hps.get('num_layers') + 1}")
print(f"Learning Rate: {best_hps.get('learning_rate')}")

# 4. Train Best Tuned Model
tuned_model = tuner.hypermodel.build(best_hps)
history_tuned = tuned_model.fit(
    X_train_scaled, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.10,
    verbose=1
)

Trial 10 Complete [00h 00m 56s]
val_accuracy: 0.8343750238418579

Best val_accuracy So Far: 0.9649999737739563
Total elapsed time: 00h 08m 40s

Best Hyperparameters Found:
Layers: 3
Learning Rate: 0.001
Epoch 1/50
450/450 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.7444 - loss: 0.8573 - val_accuracy: 0.8581 - val_loss: 0.4429
Epoch 2/50
450/450 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - accuracy: 0.8793 - loss: 0.3734 - val_accuracy: 0.9081 - val_loss: 0.2869
Epoch 3/50
450/450 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9172 - loss: 0.2546 - val_accuracy: 0.9256 - val_loss: 0.2268
Epoch 4/50
450/450 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9361 - loss: 0.1932 - val_accuracy: 0.9388 - val_loss: 0.2036
Epoch 5/50
450/450 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9481 - loss: 0.1561 - val_accuracy: 0.9306 - val_loss: 0.2266
Epoch 6/50
450/450 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9568 - loss: 0.1270 - val_accuracy: 0.9381 - val_loss: 0.2166
Epoch 7/50
450/450 ━━━━━━━━━━

## Task 4: Evaluation and Comparison



In [ ]:
from sklearn.metrics import classification_report, accuracy_score, precision_recall_fscore_support

# 1. Predictions
y_pred_base = np.argmax(base_model.predict(X_test_scaled), axis=1)
y_pred_tuned = np.argmax(tuned_model.predict(X_test_scaled), axis=1)

# 2. Compute Metrics
def calculate_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted')
    return acc, prec, rec, f1

base_acc, base_prec, base_rec, base_f1 = calculate_metrics(y_test, y_pred_base)
tuned_acc, tuned_prec, tuned_rec, tuned_f1 = calculate_metrics(y_test, y_pred_tuned)

# 3. Performance Comparison Table
comparison_df = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision (Weighted)', 'Recall (Weighted)', 'F1-Score (Weighted)'],
    'Baseline Model': [base_acc, base_prec, base_rec, base_f1],
    'Tuned Model': [tuned_acc, tuned_prec, tuned_rec, tuned_f1]
})

print("\n--- Performance Comparison ---")
print(comparison_df.round(4).to_string(index=False))

# Detailed Classification Report for Tuned Model
print("\n--- Detailed Classification Report (Tuned Model) ---")
print(classification_report(y_test, y_pred_tuned, target_names=label_encoder.classes_))